In [ ]:
import anndata
import pandas as pd
from pathlib import Path

df=pd.read_csv(snakemake.input.cellwhisperer_labels)


In [ ]:
from openai import OpenAI

system_message = "I will provide you with a number of entries of terms that describe a cluster of cells. There is a confidence score next to each term - pay more attention to large scores. Please give a very short description of the cells in the cluster based on the term. Note that not all terms will be necessarily relevant for this task - just try to find the common theme of the terms and report that. If possible, describe biological concepts beyond just cell type names. Reply with less than six words!\n"

client = OpenAI(api_key="sk-...")

In [ ]:
num_clusters = len(df["leiden"].drop_duplicates())
assert num_clusters <= snakemake.params.max_num_clusters, f"too many clusters in dataset ({num_clusters}). Stopping to prevent high GPT-4 cost"

In [ ]:
results = {}
for leiden, grouped_df in df.loc[df.library != "cell_ontology_class"].groupby("leiden"):
    grouped_df=grouped_df[grouped_df["logits"]>2].sort_values(by="logits",ascending=False)[["library","term","logits"]].iloc[:50]
    terms = [
      'library: ' + ', '.join([f"{x} ({round(val,2)})" for x, val in zip(lib_df['term'].values[:4], lib_df['logits'].values[:4] )])
      for library,lib_df in grouped_df.groupby("library")
    ]
    user_message = "\n".join(terms)

    request = [
        {"role": "system", "content": system_message},
        {"role": "user", "content": user_message},
    ]

    chat_completion = client.chat.completions.create(
        messages=request,
        model="gpt-4-turbo-preview",
        max_tokens=100,
        temperature=0,  # type: ignore [reportUndefinedVariable]
    )
    response = chat_completion.choices[0].message.content

    print(response)
    results[leiden] = response


In [ ]:
series = pd.Series(results)
series.index.name = "leiden_cluster_id"
series.name = "GPT4_generated_labels"
series.to_csv(snakemake.output.curated_labels)
